# Product Knowledge Base Builder
**Flow:**
1. Mount Google Drive
2. Scrape product descriptions and Reddit discourse for each product
3. Parse pre-collected .txt review files (Amazon / Target / Walmart) and inject into knowledge base
4. Save knowledge_base.json

## 1. Install Dependencies *(run once)*

In [21]:
import subprocess, sys

pkgs = ["requests", "httpx", "beautifulsoup4", "playwright",
        "fake-useragent", "lxml", "pandas", "openpyxl", "tqdm", "nest_asyncio"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])
subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])
print("All dependencies ready")

All dependencies ready


## 2. Imports & Configuration

In [22]:
import json, re, time, random, logging, asyncio
from pathlib import Path
from urllib.parse import urlparse
from typing import Optional

import httpx
import pandas as pd
import requests
import nest_asyncio
from bs4 import BeautifulSoup
from fake_useragent import UserAgent
from playwright.async_api import async_playwright
from tqdm import tqdm

nest_asyncio.apply()

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

OUTPUT_JSON = "/content/drive/MyDrive/CS_510/knowledge_base.json"

MAX_REVIEWS_PER_SOURCE = 50
MIN_DELAY = 2.0
MAX_DELAY = 5.0

PLAYWRIGHT_DOMAINS = {"amazon.com", "target.com", "bestbuy.com", "walmart.com"}

ua = UserAgent()
print("Config loaded")

Config loaded


## 3. Mount Google Drive
Sign in with the Google account that has access to the CS 510 shared folder.

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
DRIVE_BASE = Path("/content/drive/My Drive/CS_510/")
EXCEL_PATH = DRIVE_BASE / "product_list.xlsx"

TXT_DIRS = {
    "amazon":  DRIVE_BASE / "txt_files" / "amazon",
    "target":  DRIVE_BASE / "txt_files" / "target",
    "walmart": DRIVE_BASE / "txt_files" / "walmart",
}

print(f"Excel exists: {EXCEL_PATH.exists()}")
for source, path in TXT_DIRS.items():
    files = list(path.glob("*.txt")) if path.exists() else []
    print(f"  {source}/ -> {len(files)} txt files found")

Excel exists: True
  amazon/ -> 10 txt files found
  target/ -> 10 txt files found
  walmart/ -> 10 txt files found


In [25]:
DRIVE_BASE = Path("/content/drive/My Drive/CS_510/")
# EXCEL_PATH = DRIVE_BASE / "Product_List.xlsx"

TXT_DIRS = {
    "amazon":  DRIVE_BASE / "txt_files" / "amazon",
    "target":  DRIVE_BASE / "txt_files" / "target",
    "walmart": DRIVE_BASE / "txt_files" / "walmart",
}

# Print everything in DRIVE_BASE recursively
print(f"=== Contents of {DRIVE_BASE} ===")
if DRIVE_BASE.exists():
    for item in sorted(DRIVE_BASE.rglob("*")):
        depth = len(item.relative_to(DRIVE_BASE).parts) - 1
        indent = "  " * depth
        marker = "/" if item.is_dir() else ""
        print(f"{indent}{item.name}{marker}")
else:
    print("  [DRIVE_BASE does not exist!]")

print()
print(f"Excel exists: {EXCEL_PATH.exists()}")
for source, path in TXT_DIRS.items():
    files = list(path.glob("*.txt")) if path.exists() else []
    print(f"  {source}/ -> {len(files)} txt files found")

=== Contents of /content/drive/My Drive/CS_510 ===
Copy of CS510 Project Proposal.gdoc
Copy of kb_colab.ipynb
Copy of knowledge_base.json
Copy of scrape_knowledge_base.ipynb
Product_List.gsheet
knowledge_base.json
product_list.xlsx
txt_files/
  amazon/
    airpods4_amazon.txt
    candle_amazon.txt
    coffemaker_amazon.txt
    crocs_amazon.txt
    electrictoothbrush_amazon.txt
    owala_amazon.txt
    oxiclean_amazon.txt
    pimplepatches_amazon.txt
    proteinbar_amazon.txt
    vacuum_amazon.txt
  target/
    airpods4_target.txt
    candle_target.txt
    coffeemaker_target.txt
    crocs_target.txt
    electrictoothbrush_target.txt
    owala_target.txt
    oxiclean_target.txt
    pimplepatches_target.txt
    proteinbar_target.txt
    vacuum_target.txt
  walmart/
    airpods4_walmart.txt
    candle_walmart.txt
    coffeemaker_walmart.txt
    crocs_walmart.txt
    electrictoothbrush_walmart.txt
    owala_walmart.txt
    oxiclean_walmart.txt
    pimplepatches_walmart.txt
    proteinbar_wa

## 4. Utility Helpers

In [26]:
def get_domain(url: str) -> str:
    try:
        host = urlparse(url).netloc.lower()
        parts = host.split(".")
        return ".".join(parts[-2:]) if len(parts) >= 2 else host
    except Exception:
        return ""

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()

def polite_sleep():
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

def _rating(text: str) -> Optional[float]:
    m = re.search(r"(\d+\.?\d*)", text or "")
    return float(m.group(1)) if m else None

print("Helpers defined")

Helpers defined


## 5. Fetchers

In [27]:
HEADERS_BASE = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "DNT": "1",
}


def fetch_requests(url: str, timeout: int = 15) -> Optional[str]:
    headers = {**HEADERS_BASE, "User-Agent": ua.random}
    try:
        resp = requests.get(url, headers=headers, timeout=timeout, allow_redirects=True)
        if resp.status_code == 200:
            return resp.text
        log.warning(f"HTTP {resp.status_code} -> {url[:60]}")
    except Exception as e:
        log.warning(f"requests error: {e}")
    return None


async def _fetch_playwright_async(url: str, wait_seconds: int = 6) -> Optional[str]:
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(
                headless=True,
                args=["--disable-blink-features=AutomationControlled",
                      "--no-sandbox", "--disable-dev-shm-usage"],
            )
            ctx = await browser.new_context(
                user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
                viewport={"width": 1280, "height": 900},
                locale="en-US",
                extra_http_headers={"Accept-Language": "en-US,en;q=0.9"},
            )
            await ctx.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            page = await ctx.new_page()
            await page.route("**/*.{png,jpg,jpeg,gif,svg,woff,woff2}", lambda r: r.abort())
            await page.goto(url, timeout=45000, wait_until="domcontentloaded")
            await page.wait_for_timeout(wait_seconds * 1000)
            html = await page.content()
            await browser.close()
            return html
    except Exception as e:
        log.warning(f"Playwright error: {e}")
    return None


def fetch_playwright(url: str, wait_seconds: int = 6) -> Optional[str]:
    return asyncio.get_event_loop().run_until_complete(_fetch_playwright_async(url, wait_seconds))


def fetch_reddit_api(url: str) -> Optional[dict]:
    try:
        json_url = url.rstrip("/") + ".json?limit=100"
        headers  = {**HEADERS_BASE, "User-Agent": "Mozilla/5.0 (research-scraper/1.0)"}
        resp     = httpx.get(json_url, headers=headers, follow_redirects=True, timeout=15)
        log.info(f"  Reddit API -> HTTP {resp.status_code}")
        if resp.status_code != 200:
            return None
        data        = resp.json()
        post        = data[0]["data"]["children"][0]["data"]
        comments    = data[1]["data"]["children"]
        description = clean_text(post.get("title", "") + " " + (post.get("selftext", "") or ""))
        reviews     = []

        def extract_comments(children):
            for child in children:
                d    = child.get("data", {})
                body = clean_text(d.get("body", ""))
                if body and len(body) > 30 and body not in ("[deleted]", "[removed]"):
                    reviews.append({"title": "", "body": body[:1000], "rating": None, "source": "reddit"})
                    if MAX_REVIEWS_PER_SOURCE and len(reviews) >= MAX_REVIEWS_PER_SOURCE:
                        return
                replies = d.get("replies", "")
                if isinstance(replies, dict):
                    extract_comments(replies.get("data", {}).get("children", []))

        extract_comments(comments)
        log.info(f"    Reddit parsed: {len(reviews)} comments, desc {len(description)} chars")
        return {"description": description, "reviews": reviews}
    except Exception as e:
        log.warning(f"Reddit API error: {e}")
        return None


def fetch_amazon_reviews(url: str) -> Optional[dict]:
    m        = re.search(r'/(?:dp|gp/product)/([A-Z0-9]{10})', url)
    asin     = m.group(1) if m else None
    page_url = f"https://www.amazon.com/dp/{asin}?th=1" if asin else url
    html     = fetch_playwright(page_url, wait_seconds=8)
    if not html:
        return None
    soup        = BeautifulSoup(html, "lxml")
    description = ""
    bullets     = soup.select("#feature-bullets li span.a-list-item")
    if bullets:
        description = " ".join(clean_text(b.get_text()) for b in bullets)
    reviews = []
    for r in soup.select("div[data-hook='review']"):
        title_el  = r.select_one("a[data-hook='review-title'] span:not(.a-icon-alt)")
        body_el   = r.select_one("span[data-hook='review-body'] span")
        rating_el = r.select_one("i[data-hook='review-star-rating'] span, i[data-hook='cmps-review-star-rating'] span")
        body   = clean_text(body_el.get_text())  if body_el   else ""
        title  = clean_text(title_el.get_text()) if title_el  else ""
        rating = _rating(clean_text(rating_el.get_text())) if rating_el else None
        if body:
            reviews.append({"title": title, "body": body, "rating": rating, "source": "amazon"})
    log.info(f"    Amazon: {len(reviews)} reviews")
    return {"description": description, "reviews": reviews}


def fetch_target(url: str) -> Optional[dict]:
    html = fetch_playwright(url, wait_seconds=4)
    description = ""
    if html:
        soup = BeautifulSoup(html, "lxml")
        for sel in ["[data-test='item-details-description']",
                    "div[data-test='item-description']",
                    "div[class*='DescriptionContainer']",
                    "meta[name='description']"]:
            el = soup.select_one(sel)
            if el:
                description = clean_text(el.get("content", "") or el.get_text())
                break
    return {"description": description, "reviews": []}


def fetch(url: str):
    domain = get_domain(url)
    if   "reddit.com" in domain: return fetch_reddit_api(url)
    elif "amazon.com" in domain: return fetch_amazon_reviews(url)
    elif "target.com" in domain: return fetch_target(url)
    elif any(d in domain for d in PLAYWRIGHT_DOMAINS): return fetch_playwright(url)
    return fetch_requests(url)

print("Fetchers defined")

Fetchers defined


## 6. HTML Parser

In [28]:
def _parse_ldjson_reviews(soup, source_label: str) -> list:
    reviews = []
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data  = json.loads(script.string or "")
            items = data if isinstance(data, list) else [data]
            for item in items:
                reviews_raw = item.get("review") or item.get("reviews") or []
                if isinstance(reviews_raw, dict): reviews_raw = [reviews_raw]
                for r in reviews_raw[:MAX_REVIEWS_PER_SOURCE]:
                    body  = clean_text(str(r.get("reviewBody", "")))
                    title = clean_text(str(r.get("name", "")))
                    ri    = r.get("reviewRating", {})
                    try:
                        rating = float(ri.get("ratingValue")) if isinstance(ri, dict) and ri.get("ratingValue") else None
                    except (TypeError, ValueError):
                        rating = None
                    if body:
                        reviews.append({"title": title, "body": body, "rating": rating, "source": source_label})
        except Exception:
            pass
    return reviews


def parse_generic(soup, domain):
    result = {"description": "", "reviews": []}
    og   = soup.find("meta", attrs={"property": "og:description"})
    meta = soup.find("meta", attrs={"name": "description"})
    if og and og.get("content"):
        result["description"] = clean_text(og["content"])
    elif meta and meta.get("content"):
        result["description"] = clean_text(meta["content"])
    else:
        paras = [clean_text(p.get_text()) for p in soup.find_all("p") if len(p.get_text(strip=True)) > 40]
        result["description"] = " ".join(paras[:5])
    result["reviews"].extend(_parse_ldjson_reviews(soup, domain))
    if not result["reviews"]:
        for sel in [".review-text", ".reviewText", ".review-body",
                    "[class*='review-content']", "[itemprop='reviewBody']"]:
            for el in soup.select(sel)[:MAX_REVIEWS_PER_SOURCE]:
                text = clean_text(el.get_text())
                if len(text) > 30:
                    result["reviews"].append({"title": "", "body": text, "rating": None, "source": domain})
            if result["reviews"]: break
    return result


def parse(html: str, url: str) -> dict:
    return parse_generic(BeautifulSoup(html, "lxml"), get_domain(url))

print("Parsers defined")

Parsers defined


## 7. TXT Review File Parsers

In [ ]:
def parse_amazon_txt(text: str) -> list:
    reviews = []
    blocks  = re.split(r'\n(?=\S.*\n\d\.0 out of 5 stars)', text.strip())
    for block in blocks:
        lines = [l.strip() for l in block.strip().splitlines() if l.strip()]
        if len(lines) < 3: continue
        person   = lines[0]
        rating_m = re.search(r'(\d\.?\d*) out of 5 stars', lines[1])
        rating   = float(rating_m.group(1)) if rating_m else None
        title    = re.sub(r'^\d\.?\d* out of 5 stars\s*', '', lines[1])
        noise    = re.compile(r'^(Reviewed in|Verified Purchase|Helpful|Report|\d+ people|Size:|Color:|Customer image)', re.I)
        body_lines = [l for l in lines[2:] if not noise.match(l) and not re.match(r'^\d\.0 out of 5', l)]
        body = " ".join(body_lines).strip()
        if body:
            reviews.append({"person": person, "title": title, "body": body, "rating": rating, "source": "amazon_txt"})
    return reviews


def parse_target_txt(text: str) -> list:
    reviews = []
    blocks  = re.split(r'(?=\d out of 5 stars)', text.strip())
    for block in blocks:
        lines = [l.strip() for l in block.strip().splitlines() if l.strip()]
        if not lines: continue
        rating_m = re.search(r'(\d) out of 5', lines[0])
        rating   = float(rating_m.group(1)) if rating_m else None
        noise = re.compile(
            r'^(\d out of 5|originally posted|Report review|Did you|Thumbs|Would|Guest review'
            r'|comfort|size|style|width|\d+\.0 out of \d)', re.I)
        clean = [l for l in lines[1:]
                 if not noise.match(l)
                 and not re.match(r'^\d+ (guests?|people)', l)
                 and not re.match(r'^\w+ \d+, \d{4}', l)
                 and not re.match(r'^\d+ \w+ \d{4}', l)]
        if len(clean) < 2: continue
        person = clean[-1] if len(clean[-1]) < 30 and not clean[-1].endswith('.') else ""
        title  = clean[0]
        body   = " ".join(clean[1:-1] if person else clean[1:]).strip()
        if body:
            reviews.append({"person": person, "title": title, "body": body, "rating": rating, "source": "target_txt"})
    return reviews


def parse_walmart_txt(text: str) -> list:
    reviews = []
    blocks  = re.split(r'(?=(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d+,\s+\d{4})', text.strip())
    for block in blocks:
        lines = [l.strip() for l in block.strip().splitlines() if l.strip()]
        if len(lines) < 4: continue
        person   = lines[1] if len(lines) > 1 else ""
        rating_m = re.search(r'(\d) out of 5', block)
        rating   = float(rating_m.group(1)) if rating_m else None
        noise    = re.compile(
            r'^(Item details|Occasion:|Shoe size|Color:|Verified Purchase|Helpful|Report'
            r'|Walmart Associate|Top Reviewer|Incentivized Review|\(\d+\)|\d+ out of 5)',
            re.I)
        date_re  = re.compile(r'^(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d+', re.I)
        sold_re  = re.compile(r'^Sold and shipped by', re.I)
        clean    = [l for l in lines[2:] if not noise.match(l) and not date_re.match(l)
                    and not sold_re.match(l)
                    and not re.match(r'^Image \d+', l) and not re.match(r'^View more', l)]
        if not clean: continue
        title = clean[0]
        body  = " ".join(clean[1:]).strip() or title
        reviews.append({"person": person, "title": title, "body": body, "rating": rating, "source": "walmart_txt"})
    return reviews


def parse_txt_file(filename: str) -> list:
    """Find file in the correct Drive subfolder and parse it."""
    for source, folder in TXT_DIRS.items():
        filepath = folder / filename
        if filepath.exists():
            text = filepath.read_text(encoding="utf-8", errors="ignore")
            if source == "amazon":  return parse_amazon_txt(text)
            if source == "target":  return parse_target_txt(text)
            if source == "walmart": return parse_walmart_txt(text)
    log.warning(f"  Not found in any subfolder: {filename}")
    return []

print("TXT parsers defined")

## 8. Single-URL Scraper

In [ ]:
def scrape_url(url: str) -> dict:
    if not url or not str(url).startswith("http"):
        return {"description": "", "reviews": [], "review_count": 0, "error": "invalid or missing URL"}
    log.info(f"  Fetching: {url[:80]}")
    raw = fetch(url)
    polite_sleep()
    if raw is None:
        return {"description": "", "reviews": [], "review_count": 0, "error": "fetch failed"}
    result = raw if isinstance(raw, dict) else parse(raw, url)
    result["url"]          = url
    result["review_count"] = len(result.get("reviews", []))
    log.info(f"    desc: {len(result.get('description',''))} chars | reviews: {result['review_count']}")
    return result

print("scrape_url() defined")

## 9. Load Product List from Excel

In [ ]:
df = pd.read_excel(EXCEL_PATH, sheet_name="df_input", dtype=str).fillna("")
df.columns = [c.strip() for c in df.columns]
print(f"Loaded {len(df)} rows | Columns: {list(df.columns)}")

# Group web links by product: {product_name: {label: url}}
products_grouped = {}
for _, row in df.iterrows():
    name  = row["PRODUCT NAME"].strip()
    label = row["LABEL"].strip()
    link  = row["LINK"].strip()
    products_grouped.setdefault(name, {})[label] = link

# Map product -> set of txt filenames from the txt_file_name column
txt_files_by_product = {}
for _, row in df.iterrows():
    name     = row["PRODUCT NAME"].strip()
    txt_name = row.get("txt_file_name", "").strip()
    if txt_name:
        txt_files_by_product.setdefault(name, set()).add(txt_name)

print(f"Found {len(products_grouped)} unique products")
print(f"Products with txt files: {sum(1 for v in txt_files_by_product.values() if v)}")
df.head(10)

## 10. Phase 1 - Scrape Descriptions and Reddit
Set TEST_MODE = False to run all products.

In [15]:
TEST_MODE   = False
SAMPLE_SIZE = 2
DESC_PRIORITY = ["Amazon Link", "Actual Product", "Target", "Reddit / Reviews"]

knowledge_base = {}
to_scrape = list(products_grouped.items())[:SAMPLE_SIZE] if TEST_MODE else list(products_grouped.items())
print(f"Scraping {'TEST: ' + str(SAMPLE_SIZE) if TEST_MODE else 'ALL ' + str(len(to_scrape))} products...\n")

for product_name, links in tqdm(to_scrape, desc="Products"):
    print(f"\n{'='*60}\n> {product_name}\n{'='*60}")
    kb_entry = {
        "product_name":         product_name,
        "sources":              {},
        "combined_description": "",
        "all_reviews":          [],
    }

    for label, url in links.items():
        if not url or url == "—": continue
        print(f"  [{label}]")
        scraped = scrape_url(url)
        kb_entry["sources"][label] = {
            "url":          url,
            "label":        label,
            "description":  scraped.get("description", ""),
            "reviews":      scraped.get("reviews", []),
            "review_count": scraped.get("review_count", 0),
            "error":        scraped.get("error"),
        }
        kb_entry["all_reviews"].extend(scraped.get("reviews", []))

    for lbl in DESC_PRIORITY:
        if kb_entry["sources"].get(lbl, {}).get("description"):
            kb_entry["combined_description"] = kb_entry["sources"][lbl]["description"]
            break
    if not kb_entry["combined_description"]:
        kb_entry["combined_description"] = " | ".join(
            s["description"] for s in kb_entry["sources"].values() if s.get("description")
        )

    kb_entry["total_reviews"] = len(kb_entry["all_reviews"])
    knowledge_base[product_name] = kb_entry
    print(f"  Web reviews collected: {kb_entry['total_reviews']}")

print(f"\nPhase 1 done -- {len(knowledge_base)} products scraped")

Scraping ALL 41 products...



Products:   0%|          | 0/41 [00:00<?, ?it/s]


> Amazon Echo Dot (5th Gen)
  [Amazon Link]


Browser logs:

<launching> /root/.cache/ms-playwright/chromium_headless_shell-1217/chrome-headless-shell-linux64/chrome-headless-shell --disable-field-trial-config --disable-background-networking --disable-background-timer-throttling --disable-backgrounding-occluded-windows --disable-back-forward-cache --disable-breakpad --disable-client-side-phishing-detection --disable-component-extensions-with-background-pages --disable-component-update --no-default-browser-check --disable-default-apps --disable-dev-shm-usage --disable-extensions --disable-features=AvoidUnnecessaryBeforeUnloadCheckSync,BoundaryEventDispatchTracksNodeRemoval,DestroyProfileOnBrowserClose,DialMediaRouteProvider,GlobalMediaControls,HttpsUpgrades,LensOverlay,MediaRouter,PaintHolding,ThirdPartyStoragePartitioning,Translate,AutoDeElevate,RenderDocument,OptimizationHints --enable-features=CDPScreenshotNewSurface --allow-pre-commit-input --disable-hang-monitor --disable-ipc-flooding-protection --disable-popup-blocking --disa

  [Actual Product]


...

  [Actual Product]
  [Target]


Browser logs:

<launching> /root/.cache/ms-playwright/chromium_headless_shell-1217/chrome-headless-shell-linux64/chrome-headless-shell --disable-field-trial-config --disable-background-networking --disable-background-timer-throttling --disable-backgrounding-occluded-windows --disable-back-forward-cache --disable-breakpad --disable-client-side-phishing-detection --disable-component-extensions-with-background-pages --disable-component-update --no-default-browser-check --disable-default-apps --disable-dev-shm-usage --disable-extensions --disable-features=AvoidUnnecessaryBeforeUnloadCheckSync,BoundaryEventDispatchTracksNodeRemoval,DestroyProfileOnBrowserClose,DialMediaRouteProvider,GlobalMediaControls,HttpsUpgrades,LensOverlay,MediaRouter,PaintHolding,ThirdPartyStoragePartitioning,Translate,AutoDeElevate,RenderDocument,OptimizationHints --enable-features=CDPScreenshotNewSurface --allow-pre-commit-input --disable-hang-monitor --disable-ipc-flooding-protection --disable-popup-blocking --disa

  [Reddit / Reviews]


Browser logs:

<launching> /root/.cache/ms-playwright/chromium_headless_shell-1217/chrome-headless-shell-linux64/chrome-headless-shell --disable-field-trial-config --disable-background-networking --disable-background-timer-throttling --disable-backgrounding-occluded-windows --disable-back-forward-cache --disable-breakpad --disable-client-side-phishing-detection --disable-component-extensions-with-background-pages --disable-component-update --no-default-browser-check --disable-default-apps --disable-dev-shm-usage --disable-extensions --disable-features=AvoidUnnecessaryBeforeUnloadCheckSync,BoundaryEventDispatchTracksNodeRemoval,DestroyProfileOnBrowserClose,DialMediaRouteProvider,GlobalMediaControls,HttpsUpgrades,LensOverlay,MediaRouter,PaintHolding,ThirdPartyStoragePartitioning,Translate,AutoDeElevate,RenderDocument,OptimizationHints --enable-features=CDPScreenshotNewSurface --allow-pre-commit-input --disable-hang-monitor --disable-ipc-flooding-protection --disable-popup-blocking --disa

  Web reviews collected: 0

Phase 1 done -- 41 products scraped


## 11. Phase 2 - Inject TXT Reviews from Drive

In [16]:
for product_name, txt_filenames in txt_files_by_product.items():
    if product_name not in knowledge_base:
        continue

    entry = knowledge_base[product_name]
    print(f"\n> {product_name}")

    for filename in sorted(txt_filenames):
        reviews    = parse_txt_file(filename)
        source_key = filename.replace(".txt", "")
        entry["sources"][source_key] = {
            "url":          filename,
            "label":        source_key,
            "description":  "",
            "reviews":      reviews,
            "review_count": len(reviews),
            "error":        None,
        }
        entry["all_reviews"].extend(reviews)
        print(f"  [{source_key}] -> {len(reviews)} reviews loaded")

    entry["total_reviews"] = len(entry["all_reviews"])
    print(f"  Total reviews now: {entry['total_reviews']}")

print("\nPhase 2 done")


> Amazon Echo Dot (5th Gen)
  [echodot_amazon] -> 0 reviews loaded
  Total reviews now: 0

> Neutrogena Hydro Boost Water Gel
  [coffeemaker_amazon] -> 0 reviews loaded
  Total reviews now: 0

> Nespresso Vertuo Next Coffee Machine
  [coffeemaker_target] -> 304 reviews loaded
  Total reviews now: 304

> OxiClean White Revive Laundry Whitener
  [oxiclean_amazon] -> 100 reviews loaded
  [oxiclean_target] -> 304 reviews loaded
  Total reviews now: 404

> Apple Watch Series 11
  [vacuum_amazon] -> 100 reviews loaded
  Total reviews now: 100

> Dyson V8 Cordless Vacuum
  [vacuum_target] -> 304 reviews loaded
  Total reviews now: 312

> Crocs
  [crocs_amazon] -> 99 reviews loaded
  [crocs_target] -> 303 reviews loaded
  Total reviews now: 402

> Apple Airpods 4
  [airpods4_amazon] -> 100 reviews loaded
  [airpods4_target] -> 304 reviews loaded
  Total reviews now: 404

> Owala Water Bottle
  [owala_amazon] -> 100 reviews loaded
  [owala_target] -> 304 reviews loaded
  Total reviews now: 404

## 12. Save Knowledge Base to JSON

In [17]:
out_path = Path(OUTPUT_JSON)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(knowledge_base, f, indent=2, ensure_ascii=False)

total_reviews = sum(e['total_reviews'] for e in knowledge_base.values())
print(f"Saved -> {out_path}")
print(f"   Products     : {len(knowledge_base)}")
print(f"   Total reviews: {total_reviews}")

Saved -> /content/drive/MyDrive/CS_510/knowledge_base.json
   Products     : 41
   Total reviews: 3811


## 13. Inspect Results

In [18]:
rows = []
for name, entry in knowledge_base.items():
    for label, src in entry["sources"].items():
        rows.append({
            "Product":         name,
            "Source":          label,
            "Reviews Found":   src["review_count"],
            "Has Description": bool(src["description"]),
            "Error":           src.get("error") or "",
        })
pd.DataFrame(rows)

,Product,Source,Reviews Found,Has Description,Error
0,Amazon Echo Dot (5th Gen),Amazon Link,0,False,fetch failed
1,Amazon Echo Dot (5th Gen),Actual Product,0,False,fetch failed
2,Amazon Echo Dot (5th Gen),Target,0,False,
3,Amazon Echo Dot (5th Gen),Reddit / Reviews,0,False,fetch failed
4,Amazon Echo Dot (5th Gen),echodot_amazon,0,False,
...,...,...,...,...,...
169,Yankee Candle Lemon Lavender Scented Candle,Actual Product,0,True,
170,Yankee Candle Lemon Lavender Scented Candle,Target,0,False,
171,Yankee Candle Lemon Lavender Scented Candle,Reddit / Reviews,0,False,fetch failed
172,Yankee Candle Lemon Lavender Scented Candle,candle_amazon,100,False,


In [19]:
entry = knowledge_base[list(knowledge_base.keys())[0]]
print(f"Product      : {entry['product_name']}")
print(f"Total reviews: {entry['total_reviews']}")
print(f"\nDescription:\n{entry['combined_description'][:400]}")
print(f"\nFirst 5 reviews:")
for r in entry["all_reviews"][:5]:
    person = r.get('person', '')
    print(f"  [{r['source']}] stars={r['rating']} {person} | {r['title']} -- {r['body'][:100]}")

Product      : Amazon Echo Dot (5th Gen)
Total reviews: 0

Description:


First 5 reviews:


## Debug *(optional)*

In [20]:
DEBUG_URL = "https://www.reddit.com/r/scacjdiscussion/comments/izbqbn/hot_take_cerave_moisturizing_cream_in_the_tub_is/"

raw = fetch(DEBUG_URL)
if isinstance(raw, dict):
    print(f"Pre-parsed dict -- {len(raw['reviews'])} reviews")
    print(f"Description: {raw['description'][:200]}")
    for r in raw['reviews'][:3]:
        print(f"  - {r['body'][:120]}")
else:
    soup = BeautifulSoup(raw, "lxml")
    with open("debug_output.html", "w") as f: f.write(raw)
    print(f"HTML -- {len(raw)} chars")
    print(soup.get_text()[:2000])

TypeError: Incoming markup is of an invalid type: None. Markup must be a string, a bytestring, or an open filehandle.